# OmniVoice Project Studio — Kaggle Gradio Simple (Dual-T4 optimized)

Minimal Gradio-only launcher tuned for Kaggle **2× Tesla T4** sessions.

```text
cuda:0  → OmniVoice TTS
cuda:1  → Whisper ASR verification
CPU/RAM → preprocessing + Gradio + file I/O
SSD     → /kaggle/working/OmniVoiceStudio + model cache
```

For dual-GPU sessions the notebook explicitly assigns ASR to `cuda:1`. Hardware detection remains advisory, so a stale imported module can no longer leave GPU1 idle.


In [ ]:
# Kaggle Internet must be enabled for GitHub/Hugging Face downloads.
import os
from pathlib import Path

WORKSPACE = "/kaggle/working/OmniVoiceStudio"
CACHE_ROOT = "/kaggle/working/.cache"
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%pip install -q --upgrade --no-cache-dir "git+https://github.com/binhminhanh1235/OmniVoice.git@master"


In [ ]:
import importlib
import shutil
import torch
import omnivoice.hardware_quality as hardware_quality

# Reload after pip upgrade so a previously imported detector cannot stay stale in this kernel.
hardware_quality = importlib.reload(hardware_quality)

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Notebook settings.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

hardware = hardware_quality.detect_hardware(device_index=0)
TTS_DEVICE = "cuda:0"
# Runtime topology is authoritative: never leave a healthy second T4 idle because of a stale recommendation.
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else hardware.recommended_asr_device
ASR_MODEL = "openai/whisper-small.en"

if GPU_COUNT >= 2 and hardware.recommended_asr_device != "cuda:1":
    print("WARNING: detector recommendation differs from dual-GPU runtime; forcing ASR to cuda:1.")

quality_store = hardware_quality.HardwareQualitySettingsStore(WORKSPACE)
if not quality_store.path.exists():
    quality_store.set_default(hardware.recommended_preset)
current_preset = quality_store.load().default_preset

print("Hardware profile:", hardware.summary())
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL)
print("Workspace quality preset:", current_preset)
print("Workspace:", WORKSPACE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")


## Launch Gradio

On 2×T4, OmniVoice runs on GPU0 and `whisper-small.en` runs in FP16 on GPU1. If only one GPU is available, ASR falls back to the hardware recommendation, normally CPU for a single T4.


In [ ]:
!omnivoice-project-studio \
  --model k2-fsa/OmniVoice \
  --device {TTS_DEVICE} \
  --workspace {WORKSPACE} \
  --asr-model {ASR_MODEL} \
  --asr-device {ASR_DEVICE} \
  --share
